# Soccer match event tagger — run on Google Colab

Runs the match-tagger web UI on a free Colab GPU and gives you a public link to open in any browser. Nothing to install on your computer.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**
3. After 2–3 minutes the last cell prints `Running on public URL: https://xxxx.gradio.live`. Open it, upload a match video, and click **Tag match**.

The link works while this Colab tab stays open. If Colab disconnects, run everything again (you get a new link).

The model is [T-DEED](https://github.com/arturxe2/T-DEED) (`SoccerNetBall_challenge2` checkpoint, trained on SoccerNet broadcasts). See **How reliable is each class?** in the app before trusting tackles, free kicks, goals and high passes.

## 1. Settings

In [ ]:
# Use files from your Google Drive: the model in MyDrive/CS496/model and videos you can open by path.
# Leave False to download the model from the T-DEED authors and upload videos in the app.
USE_GOOGLE_DRIVE = False  # @param {type:"boolean"}
DRIVE_FOLDER = "CS496"  # @param {type:"string"}

## 2. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU, then run again"
print("GPU:", torch.cuda.get_device_name(0))

## 3. Install the app and the model (1–3 minutes)

In [ ]:
import shutil, subprocess
from pathlib import Path
%cd /content
!rm -rf TEED-match-tagger
!git clone -q --branch cs496-evaluation-and-reliability https://github.com/jmoussa99/TEED-match-tagger.git
!git clone -q https://github.com/arturxe2/T-DEED TEED-match-tagger/private/T-DEED
!git -C TEED-match-tagger/private/T-DEED checkout -q d953b4de8a2ad941c565e1683c5b07255cbe644e
!git -C TEED-match-tagger/private/T-DEED apply /content/TEED-match-tagger/app/patches/tdeed_mps.patch
!pip -q install gradio==6.31.0 timm gdown

CKPT = Path("/content/TEED-match-tagger/private/T-DEED/checkpoints/SoccerNetBall/SoccerNetBall_challenge2/checkpoint_best.pt")
CKPT.parent.mkdir(parents=True, exist_ok=True)
EXTRA = []
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    EXTRA = ["--extra-path", "/content/drive"]
    mine = Path("/content/drive/MyDrive") / DRIVE_FOLDER / "model" / "checkpoint_best.pt"
    if mine.is_file():
        shutil.copy(mine, CKPT); print("Model copied from your Google Drive")
if not CKPT.is_file():
    subprocess.run(["gdown", "-q", "1uCWam8EhAXt3z6QQAfhRfzHITUcml438", "-O", str(CKPT)], check=True)
    print("Model downloaded from the T-DEED authors")
print(f"Model: {CKPT.stat().st_size / 1e6:.0f} MB")
if USE_GOOGLE_DRIVE:
    vids = sorted((Path("/content/drive/MyDrive") / DRIVE_FOLDER).rglob("*.mp4"))
    print("Videos you can paste into 'local file path':", *[str(v) for v in vids] or ["(none found)"], sep="\n  ")

## 4. Start the app

Wait for **`Running on public URL: https://xxxx.gradio.live`** and open it. This cell keeps running while the app is up; stop it to shut the app down.

- **Upload** a match video in the app, or, with Google Drive on, paste a path such as `/content/drive/MyDrive/CS496/demo_videos/gar_demo_1.mp4` into **local file path** (faster for large files).
- A full 90-minute match takes roughly 10–20 minutes on a T4.

In [ ]:
%cd /content/TEED-match-tagger
!python scripts/launch_colab.py {" ".join(EXTRA)}